In [ ]:
# =========================================================================
# Qwen-Image-2.1-Turbo A/B 橫向對比實測 Worker (同一套黃金 Prompt 測試)
# =========================================================================
import os
import json
import time
import random
import shutil

# ==================== 1. 絕對鎖定的核心人設 (四肢纖細精實、沙漏極致火辣、極致冷白牛奶肌、自然重磅柔軟 F 罩杯大胸深事業線) ====================
CORE_PERSONA = (
    "Intimate waist-up boudoir photo of three distinct breathtakingly gorgeous 20yo French-Vietnamese mixed-race Eurasian beauties with unique angelic faces, "
    "wearing alluring deep-V sheer tulle mesh and floral lace bodysuits (floral lace artfully covering bust apex, non-nude), "
    "extremely fair pale luminous milk-white skin tone, healthy soft rosy cheeks, authentic tender skin texture with visible micro-pores, "
    "slender toned limbs, slim graceful arms, delicate collarbones, flat toned stomach, tiny narrow waist, "
    "voluptuous curvy hourglass proportions, ((voluptuous large natural heavy soft F-cup breasts:1.4)), ((prominent deep cleavage:1.3)), (plump full bust:1.3)."
)

# 雙重防線：
# 1) 嚴格封鎖露點/全裸/違規字眼 (保護帳號 100% 安全不越界)
# 2) 嚴格封鎖過肥、臃腫、粗手臂、粗腰 (fat, chubby, overweight, flabby, thick arms, thick waist, plus-size)
# 3) 嚴格封鎖過壯、肌肉金剛、虎背熊腰 (muscular, bodybuilder, buff, broad masculine shoulders, masculine physique)
# 4) 嚴格封鎖營養不良、排骨肋骨、乾癟 (malnourished, skinny, bony, visible ribs, gaunt, hollow cheeks)
# 5) 嚴格封鎖小胸、平胸、縮胸 (small breasts:1.4, flat chest:1.4, modest bust:1.3, petite breasts:1.3)
# 6) 嚴格封鎖黑皮、曬黑、黃黑皮 (dark skin, tanned skin, brown skin, sunburn, dusky)
# 7) 嚴格封鎖塑料人偶、陶瓷磨皮、美顏濾鏡 (porcelain, plastic, mannequin, doll, airbrushed, silicone)
# 8) 嚴格封鎖眼歪嘴斜、瞳孔不對稱 (asymmetrical eyes, crooked eyes, misaligned pupils, crooked mouth)
# 9) 嚴格封鎖多頭、額外人頭、撞臉/複製人臉 (extra heads, 4 persons, clone faces)
# 10) 嚴格封鎖手指畸變、多手指 (bad hands, missing fingers, extra fingers, mutated hands)
DEFAULT_NEGATIVE_PROMPT = (
    "nude, topless, naked, exposed nipples, areola, "
    "fat, chubby, obese, flabby, thick arms, muscular, bodybuilder, "
    "flat chest, small breasts, "
    "4 persons, extra heads, extra faces, extra limbs, "
    "dark skin, tanned, sunburn, "
    "plastic skin, mannequin, doll, airbrushed, porcelain, "
    "bad hands, missing fingers, extra fingers, "
    "distorted eyes, asymmetrical eyes, crooked mouth, bad anatomy"
)

# ==================== 2. 影片生成專用：合規性感「換裝 / 褪下外罩衫 / 整理蕾絲肩帶」動態 Prompt 池 ====================
VIDEO_CHANGING_CLOTHES_PROMPTS = [
    {
        "zh": "【合規換裝動感 A】緩緩將絲質晨袍從雙肩褪下，露出內搭深 V 透膚蕾絲連身衣、雙手輕調細肩帶與自然呼吸律動",
        "en": (
            "Three young Eurasian women in a sunlit bedroom changing clothes, slowly slipping loose silk morning robes "
            "off their shoulders to reveal alluring deep-V floral lace bodysuits underneath (remaining covered in elegant lace lingerie), "
            "gently adjusting their thin lace straps with their hands, natural soft chest breathing and upper body movement, "
            "warm alluring smiles at the camera, photorealistic live-action human motion, high skin detail."
        )
    },
    {
        "zh": "【合規換裝動感 B】在全身鏡前試穿極薄網紗蕾絲內衣、手指輕拉調整肩帶與腰身蕾絲、撥弄長捲髮並轉身展現沙漏曲線",
        "en": (
            "Three young Eurasian women wearing sheer tulle mesh and floral lace lingerie sets in front of a large mirror, "
            "gently pulling up and fitting delicate lace straps over their shoulders as they finish dressing, brushing long wavy hair back "
            "with their fingers, turning their torsos naturally to admire their hourglass curves in the mirror, realistic body movement and soft eye blinks."
        )
    },
    {
        "zh": "【合規換裝動感 C】解開外層透視雪紡罩衫換裝、展示內層精緻蕾絲馬甲與深邃事業線、在純白床單上親密微傾互動",
        "en": (
            "Three young Eurasian women sitting closely on crisp white bedsheets changing outfits, "
            "slowly untying and slipping off an outer sheer chiffon wrap to showcase their delicate floral lace bralettes and bodysuits underneath, "
            "leaning naturally toward each other with soft smiles, realistic hand gestures smoothing their lace lingerie, natural breathing and subtle body sway."
        )
    }
]

DEFAULT_VIDEO_NEGATIVE_PROMPT = (
    "nude, nudity, naked, topless, exposed nipples, areola, explicit, nsfw, stripping naked, "
    "static image, camera shake only, frozen body, stiff pose, t-shirt, street clothes, outdoors, "
    "distorted face, morphing features, extra limbs, deformed hands, blurry, low quality, watermark"
)

# ==================== 3. 帶中文標籤的隨機多樣化維度池 (精準踩在 ref_1 / ref_2 性感邊界、絕不露點越界) ====================
FACE_SHAPES = [
    {"zh": "法式深邃立體混血鵝蛋臉（高挺混血鼻樑、微翹厚唇）", "en": "distinct Eurasian French-Vietnamese face with high bridge nose, full lips, deep-set hazel eyes"},
    {"zh": "清純甜美法亞心形臉（精緻下巴、大雙眼皮深邃眼眸）", "en": "distinct heart-shaped Eurasian face with delicate chin, large expressive double-eyelid almond eyes"},
    {"zh": "成熟優雅立體法亞 V 臉（高顴骨、柔和混血輪廓）", "en": "distinct sculptured Eurasian face with high cheekbones and elegant soft French jawline"},
    {"zh": "精靈感混血娃娃臉（微圓雙頰、俏皮法式微翹唇）", "en": "distinct cute Eurasian baby-face with naturally plump cheeks and subtle French pout"},
    {"zh": "高貴混血菱形輪廓（深邃眼窩、立體五官線條）", "en": "distinct diamond-shaped Eurasian face with striking high cheekbones and defined features"},
    {"zh": "清冷氣質法亞混血輪廓（立體下顎線、迷人法式混血眼妝）", "en": "distinct chic Eurasian face with sharp defined jawline, captivating French-Asian facial geometry"}
]

HAIR_STYLES = [
    {"zh": "大波浪長捲髮", "en": "long flowing wavy hair"},
    {"zh": "及腰絲滑長直髮", "en": "silky straight waist-length hair"},
    {"zh": "法式下巴短鮑伯頭", "en": "chic chin-length French bob haircut"},
    {"zh": "隨性高丸子頭配碎髮", "en": "effortless messy high bun with loose face-framing tendrils"},
    {"zh": "鎖骨層次中長髮", "en": "shoulder-length layered collarbone cut"},
    {"zh": "浪漫半紮公主波浪捲", "en": "long romantic half-up waves"},
    {"zh": "空氣瀏海低馬尾", "en": "sleek low ponytail with wispy bangs"}
]

HAIR_COLORS = [
    {"zh": "純粹烏黑", "en": "jet black"},
    {"zh": "深濃縮咖啡棕", "en": "deep espresso brown"},
    {"zh": "暖栗褐", "en": "warm chestnut brown"},
    {"zh": "黑巧克力棕", "en": "dark chocolate brown"},
    {"zh": "焦糖冷灰棕", "en": "caramel ash brown"},
    {"zh": "蜂蜜紅棕", "en": "rich honey auburn"},
    {"zh": "柔霧黑棕", "en": "soft black-brown"}
]

MAKEUP_STYLES = [
    {"zh": "水光裸妝透亮唇", "en": "dewy bare-faced no-makeup look with glossy lips"},
    {"zh": "法式玫瑰腮紅配微雀斑", "en": "soft French rose tint on lips and cheeks with subtle freckles"},
    {"zh": "蜜桃水光玻璃肌", "en": "luminous glass-skin finish with peach-pink blush"},
    {"zh": "自然細眼線配珊瑚粉唇", "en": "delicate natural winged eyeliner and soft coral-pink lips"},
    {"zh": "晨間自然紅潤透亮肌", "en": "fresh natural rosy flush with clear hydrated skin"}
]

ATTIRE_STYLES = [
    {
        "zh": "【深 V 透膚網紗蕾絲連身衣】全透視薄紗腰身配精緻立體刺繡罩杯與高叉剪裁（對齊 ref_1 款式，性感不露點）",
        "category": "plunge_sheer_bodysuit",
        "needs_color": True,
        "en": "alluring deep-V plunging sheer tulle mesh and floral Chantilly lace bodysuit with high-cut leg openings, translucent mesh midriff and delicate floral lace embroidery covering the bust cups, showcasing full F-cup cleavage"
    },
    {
        "zh": "【法式薄紗蕾絲低胸內衣組】半透明網紗托胸內衣配細帶蕾絲低腰褲（對齊 ref_2 款式，凸顯事業線與平坦小腹）",
        "category": "sheer_lace_bralette_set",
        "needs_color": True,
        "en": "delicate sheer tulle mesh and floral lace demi-cup bralette with thin spaghetti straps and matching sheer lace low-rise bikini brief, artfully covering the bust apex while accentuating deep F-cup cleavage and toned waist"
    },
    {
        "zh": "【微透膚綁帶蕾絲馬甲連身衣】深 V 爆乳剪裁配全透明網紗側腰與細繩高叉設計",
        "category": "strappy_sheer_corset",
        "needs_color": True,
        "en": "sensual deep-plunge sheer mesh and scalloped floral lace corset bodysuit with thin strappy waist details and high-cut hips, floral lace cups covering the bust apex while revealing dramatic cleavage"
    },
    {
        "zh": "【半褪絲質薄紗罩衫配蕾絲內衣】肩頭微滑落的透明薄紗睡袍，內搭深 V 蕾絲三點式內衣組",
        "category": "silk_robe_and_lace_set",
        "needs_color": True,
        "en": "translucent gossamer chiffon negligee slipping slightly off her shoulders over a fitted deep-V floral lace bralette and sheer mesh panties, highlighting her full F-cup cleavage and hourglass curves"
    }
]

LINGERIE_COLORS = [
    {"zh": "珍珠白", "en": "pure pearl white"},
    {"zh": "午夜黑", "en": "midnight black"},
    {"zh": "勃根地深酒紅", "en": "deep burgundy wine red"},
    {"zh": "香檳裸膚色", "en": "champagne blush nude"},
    {"zh": "乾燥玫瑰粉", "en": "muted dusty rose pink"},
    {"zh": "柔霧薰衣草紫", "en": "soft lavender mist"},
    {"zh": "冰川淺藍", "en": "pale ice blue"},
    {"zh": "香檳象牙白", "en": "champagne ivory"}
]

POSES_AND_INTERACTIONS = [
    {
        "zh": "【趴姿撐肘性感特寫】趴在純白柔軟床單上、雙肘自然撐在羽絨枕頭上微托下巴，胸口受重力自然微擠壓展現傲人重磅 F 罩杯深邃事業線，抬眸甜美看鏡頭",
        "short_en": "lying prone on white bed propped on elbows, deep cleavage compressed against mattress, looking up",
        "en": "sensual prone pose lying on their stomachs on plush white bedsheets, propped up gracefully on their elbows and forearms, soft breasts naturally compressed against the mattress creating dramatic deep F-cup cleavage, looking up captivatingly into the camera with soft alluring smiles"
    },
    {
        "zh": "【正面仰躺親密特寫】親密並排正面仰躺在純白羽絨床單上，雙手自然舒展搭在枕邊，長髮絲滑散開在白色枕面上，傲人 F 罩杯自然柔軟微溢，直視鏡頭",
        "short_en": "lying on backs closely together on white bed, long hair spread on pillows, direct eye contact",
        "en": "intimate overhead view of the three women lying comfortably on their backs closely together on soft white linen bedsheets, silky hair spread like a dark fan across white pillows, large natural soft F-cup breasts resting softly with deep natural cleavage, gazing alluringly directly at the camera"
    },
    {
        "zh": "【優雅側臥曼妙 S 曲線】優雅側臥在蓬鬆羽絨被上，單手微枕頭部、另一手自然平放床面，極致沙漏身材與爆乳 F 罩杯在側光下一覽無遺",
        "short_en": "side-lying gracefully on soft white duvet, resting head on arm, direct gaze",
        "en": "sensual side-lying pose reclining gracefully on a plush white duvet, one arm propping up the head, the other resting gently flat on the linen, showcasing dramatic hourglass hip-to-waist curves, soft heavy F-cup breasts, and enchanting direct gaze"
    },
    {
        "zh": "【床沿微傾親密依偎】坐在柔軟床沿上半身親密微傾朝鏡頭，雙手自然平放於膝上身側，深 V 薄紗透膚蕾絲垂墜勾勒深邃事業線與白皙美肩",
        "short_en": "sitting on bed edge leaning forward together, hands on knees, gentle smiles",
        "en": "sitting closely along the edge of a soft white bed leaning gently forward toward the camera, hands resting naturally flat on their thighs, deep plunging V-neck floral lace bodices revealing spectacular F-cup cleavage and delicate porcelain shoulders"
    },
    {
        "zh": "【高低錯落親密互動】三人高低層次錯落：一人側臥床邊、一人趴姿撐肘托腮、一人微靠床頭，生動自然且極具層次感，三人神情甜美各異",
        "short_en": "dynamic boudoir scene on white bed, one side-lying, one prone on elbows, one seated",
        "en": "dynamic multi-level boudoir composition on crisp white bedding, one woman lounging on her side, one lying prone propped on elbows, and one sitting leaning back on soft pillows, each showing different flattering angles of their natural heavy F-cup curves and sheer floral lace bodysuits"
    }
]

BACKGROUNDS_AND_LIGHTING = [
    {
        "zh": "日式傳統和室榻榻米 ＋ 背後大型全身鏡清晰反射美背與全身曲線 ＋ 障子紙窗自然柔光（原始 Prompt 場景）",
        "short_en": "in a Japanese washitsu tatami room with a large mirror behind them reflecting their backs, shoji window light",
        "en": "on Japanese tatami mats in a traditional washitsu room with a large clear mirror behind them clearly reflecting their full backs and bodies, natural window light from shoji screens"
    },
    {
        "zh": "巴黎極簡純白床單臥室 ＋ 暖白牆面與側邊柔和晨光（完全對齊 ref_1 & ref_2 參考圖場景）",
        "short_en": "on crisp white linen bedsheets in a minimalist bedroom against a clean warm-white wall, soft morning window light",
        "en": "on crisp white linen bedsheets in a minimalist sunlit bedroom against a warm off-white wall, soft diffused morning window light casting gentle shadows"
    },
    {
        "zh": "北歐落地窗陽光臥室 ＋ 無框大型全身鏡反射背影 ＋ 黃金時刻柔光",
        "short_en": "in a bright bedroom studio with a large full-length mirror reflecting their backs, golden-hour window light",
        "en": "in a bright Scandinavian bedroom with floor-to-ceiling windows and a large full-length frameless mirror reflecting their backs, bathed in soft golden-hour sunlight"
    }
]


def generate_diversified_prompt(seed=None, **kwargs):
    """
    產生「精準踩在性感邊界、100% 合規不露點、且不超過 SDXL 77-Token 視窗」的雙層寫實 Prompt。
    """
    rng = random.Random(seed)
    image_seed = seed if seed is not None else rng.randint(10000000, 99999999)

    faces = rng.sample(FACE_SHAPES, 3)
    hairs = rng.sample(HAIR_STYLES, 3)
    colors = rng.sample(HAIR_COLORS, 3)
    makeups = rng.sample(MAKEUP_STYLES, 3)
    lingerie_colors = rng.sample(LINGERIE_COLORS, 3)
    attires = rng.sample(ATTIRE_STYLES, 3)

    pose = rng.choice(POSES_AND_INTERACTIONS)
    bg = rng.choice(BACKGROUNDS_AND_LIGHTING)
    vid_motion = rng.choice(VIDEO_CHANGING_CLOTHES_PROMPTS)

    women_descriptions = []
    women_metadata = []
    positions_en = ["left woman", "center woman", "right woman"]
    positions_zh = ["左邊女孩", "中間女孩", "右邊女孩"]

    short_color_list = ", ".join([c["en"] for c in lingerie_colors])

    for idx in range(3):
        att = attires[idx]
        attire_en = f"wearing a {lingerie_colors[idx]['en']} {att['en']}"
        attire_zh = f"{lingerie_colors[idx]['zh']} — {att['zh']}"

        desc = (
            f"The {positions_en[idx]} has a {faces[idx]['en']}, {colors[idx]['en']} {hairs[idx]['en']}, "
            f"{makeups[idx]['en']}, {attire_en}."
        )
        women_descriptions.append(desc)
        women_metadata.append({
            "position_zh": positions_zh[idx],
            "face_zh": faces[idx]["zh"],
            "hair_color_zh": colors[idx]["zh"],
            "hair_style_zh": hairs[idx]["zh"],
            "makeup_zh": makeups[idx]["zh"],
            "attire_zh": attire_zh,
            "category": att["category"]
        })

    # 精準壓縮在 ~65 Tokens 內，確保 100% 完整被 CLIP 吸收（杜絕靜默截斷）：
    # 1) 衣物與非裸體合規防線置於前 15 Tokens（絕對不露點）
    # 2) 姿勢置於前 25 Tokens（趴姿/仰躺/側臥 100% 生效）
    # 3) 天然重磅柔軟 F 罩杯深溝（自然受重力擠壓）
    # 4) 四肢極度纖細緊實、蜂腰平腹（嚴防發福肥胖）
    # 5) 牛奶冷白肌與微觀毛孔
    clip_short_prompt = (
        f"Intimate RAW boudoir photo of 3 distinct 20yo French-Vietnamese women "
        f"wearing sheer floral lace lingerie bodysuits (non-nude lace covering apex), "
        f"{pose['short_en']}, natural heavy soft F-cup breasts, prominent deep cleavage, "
        f"slender toned limbs, slim graceful arms, tiny wasp waist, "
        f"milk-white luminous skin, visible microscopic skin pores, clear symmetrical eyes, "
        f"{bg['short_en']}."
    )

    full_prompt = (
        f"{clip_short_prompt} "
        f"{' '.join(women_descriptions)} "
        f"All three are {pose['en']}, {bg['en']}. "
        "Shot on 35mm f/1.8 lens, Kodak Portra 400 film, raw unedited photography, natural skin pores and soft natural breast cleavage texture, fine epidermal details, soft window light, tasteful high-fashion boudoir."
    )

    summary_lines = [
        f"🔑 隨機種子 (Seed): {image_seed}",
        f"🏞️ 場景與光線: {bg['zh']}",
        f"💃 姿勢與互動: {pose['zh']}",
        f"🎬 對應合規換裝動態 Prompt: {vid_motion['zh']}"
    ]
    for w in women_metadata:
        summary_lines.append(
            f"👩 {w['position_zh']}: {w['face_zh']} | {w['hair_color_zh']}{w['hair_style_zh']} | {w['makeup_zh']} | {w['attire_zh']}"
        )

    metadata = {
        "seed": image_seed,
        "core_persona_zh": "20歲法越混血女性 × 3、無磨皮真實毛孔質感、冷白透亮瓷肌、165cm 50kg 自然柔軟 F 罩杯沙漏體態（深 V 透膚蕾絲精準遮蔽重點、性感不越界）",
        "background_zh": bg["zh"],
        "pose_zh": pose["zh"],
        "video_motion_zh": vid_motion["zh"],
        "video_motion_prompt": vid_motion["en"],
        "video_negative_prompt": DEFAULT_VIDEO_NEGATIVE_PROMPT,
        "women": women_metadata,
        "summary_text": "\n".join(summary_lines),
        "clip_short_prompt": clip_short_prompt,
        "full_prompt": full_prompt,
        "negative_prompt": DEFAULT_NEGATIVE_PROMPT
    }
    return full_prompt, metadata




import sys, gc, glob
import torch, psutil
from PIL import Image, ImageFilter

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_name = f"{torch.cuda.get_device_name(0)} (x{gpu_count})" if gpu_count > 0 else "CPU"
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if gpu_count > 0 else 0
ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)
print(f"✅ 硬體檢測完成 | GPU: {gpu_name} | 單卡 VRAM: {vram_gb} GB | System RAM: {ram_gb} GB")

# 安裝與升級 Qwen-Image 專屬 Diffusers 與依賴
print("📦 正在確認 Qwen-Image 依賴庫...")
import subprocess
subprocess.run(["pip", "install", "-q", "diffusers>=0.41.0", "transformers>=5.17.0", "accelerate", "pillow"], check=False)

from diffusers import QwenImage21Pipeline

model_id = "Qwen/Qwen-Image-2.1-Turbo"
print(f"💎 [新世代 MMDiT 挑戰者] 正在載入頂級通義千問影像模型: {model_id}...")

# 啟用逐層序列化 CPU Offload (Sequential Offload)，確保 16GB 顯存零負擔 (峰值僅 4~6GB)
print("⚙️ 啟用逐層序列化 CPU Offload (Sequential CPU Offload)...")
pipe = QwenImage21Pipeline.from_pretrained(
    model_id, torch_dtype=torch.bfloat16
)
pipe.enable_sequential_cpu_offload()

image_times = []
image_peak_vram = 0.0
human_ts = time.strftime("%Y-%m-%d %H:%M:%S")
num_imgs = 4

# 固定種子清單進行嚴格 1:1 橫向對比
fixed_seeds = [81524489, 13838649, 99346877, 38912080]

for i in range(num_imgs):
    curr_seed = fixed_seeds[i] if i < len(fixed_seeds) else None
    full_prompt, recipe = generate_diversified_prompt(seed=curr_seed)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    gen = torch.Generator("cuda").manual_seed(recipe["seed"])
    
    # 徹底釋放 Qwen-Image-2.1 的 Qwen3-VL 多模態大語言解析力：
    # 絕不再被 77-Token 限制！直接注入包含 3 位女孩完整五官、髮型、微觀毛孔、蕾絲雕花、趴姿深溝與電影級光影的 400+ 字全細節 Full Prompt！
    print(f"🎨 [Qwen-Image 全細節長文本生成中 #{i+1}/4] Seed: {recipe['seed']} ...")
    base_img = pipe(
        prompt=recipe["full_prompt"],
        num_inference_steps=12,
        true_cfg_scale=1.0,
        generator=gen
    ).images[0]

    # 無損超取樣至 2K 超清規格
    target_w, target_h = 1440, 1850
    img_2k = base_img.resize((target_w, target_h), Image.LANCZOS)
    img_2k = img_2k.filter(ImageFilter.UnsharpMask(radius=1.2, percent=45, threshold=3))

    dt = round(time.time() - t0, 2)
    pk = round(torch.cuda.max_memory_allocated() / (1024**3), 2) if torch.cuda.is_available() else 0.0
    image_times.append(dt)
    image_peak_vram = max(image_peak_vram, pk)

    recipe["timestamp"] = human_ts
    recipe["model_engine"] = "Qwen-Image-2.1-Turbo (MMDiT + Qwen3-VL 語義理解)"
    recipe["elapsed_sec"] = dt
    recipe["peak_vram_gb"] = pk
    recipe["resolution"] = f"{target_w}x{target_h} (2K Enhanced)"

    fname = f"portrait_{i+1}.jpg"
    rname = f"portrait_{i+1}_recipe.json"
    img_2k.save(fname, quality=95)
    with open(rname, "w", encoding="utf-8") as rf:
        json.dump(recipe, rf, ensure_ascii=False, indent=2)
    print(f"   ➔ 已產出 {fname} (耗時: {dt}s, 顯存: {pk} GB)")

bench = {
    "engine": "Qwen-Image-2.1-Turbo (MMDiT + Qwen3-VL)",
    "gpu": gpu_name,
    "vram_gb": vram_gb,
    "avg_image_sec": round(sum(image_times)/len(image_times), 2) if image_times else 0,
    "peak_vram_gb": image_peak_vram,
    "count": num_imgs
}
with open("benchmark_report.json", "w", encoding="utf-8") as bf:
    json.dump(bench, bf, indent=2)

print("🎉 Qwen-Image 批次生成全數完成！")
